<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_0_3_ic_score_and_correlation_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_0_3_ic_score_and_correlation_features

## Introducción


Esta notebook integra y evalúa el conjunto de features disponibles para el índice MNQ. Se alinean los datasets de indicadores técnicos y alpha factors, se aplican criterios de selección basados en IC Score y correlación, y finalmente se prepara un dataset consolidado de entrada llamado mnq_to_model, listo para entrenar modelos predictivos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente (mnq_technical_indicators y mnq_alpha_factors) y se muestra un resumen de la información del dataset MNQ.

1. Alineación de datasets

    Se alinean horizontalmente los datasets de indicadores técnicos y alpha factors utilizando el índice temporal para garantizar correspondencia fila a fila. Se eliminan posibles columnas duplicadas, obteniendo un dataset combinado que integra toda la información disponible.

2. Criterio IC Score

    Se calculan métricas de IC Score para cada feature, representadas como el promedio y la desviación estándar del Information Coefficient respecto a los retornos objetivos. Se genera un ranking de features ordenados por la magnitud de su señal predictiva, que sirve como criterio para priorizar variables en el modelado.

3. Correlación de features

    Se construye una matriz de correlación entre los features seleccionados. El análisis permite identificar redundancia entre variables y decidir cuáles conviene descartar. Se generan tablas y visualizaciones que destacan grupos de features con alta correlación.

4. Análisis en conjunto

    Se combinan los resultados del IC Score y de la correlación. El objetivo es obtener un subconjunto de features con alta capacidad predictiva pero baja redundancia entre ellos. Este cruce asegura que los factores elegidos sean complementarios en lugar de repetitivos.

5. Preparamos el dataset mnq_to_model

    Con base en los criterios anteriores, se arma el dataset final mnq_to_model. Incluye las columnas base (OHLCV y targets) junto con el conjunto de features seleccionados. El resultado se guarda en formato Parquet y queda listo para la fase de entrenamiento de modelos.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [122]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [123]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [124]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [125]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr


### 0.4. Carga de datasets



In [126]:
def load_data(data: str):

    data_path = f'{drive_path}/5_transformer_90_model/{data}.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [127]:
def load_ic(data: str):

    data_path = f'{drive_path}/5_transformer_90_model/ic_{data}.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    return df

In [128]:
mnq_intraday_data = load_data('mnq_intraday_data')


In [129]:
mnq_technical_indicators = load_data('mnq_technical_indicators')
ic_technical_indicators = load_ic('technical_indicators')
mnq_technical_indicators.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'target_return_90',
       'rsi_14', 'rsi_7', 'momentum_10', 'momentum_5', 'macd', 'price_ema20',
       'price_ema30', 'stoch_k_20', 'stoch_k_30', 'bb_20', 'bb_30', 'bb_60',
       'atr_norm', 'roc_20', 'roc_30', 'roc_60'],
      dtype='object')

In [130]:
mnq_alpha_factors = load_data('mnq_alpha_factors')
ic_alpha_factors = load_ic ('alpha_factors')
mnq_alpha_factors.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'target_return_90',
       'rev_mom_z_90', 'rev_score_90', 'rev_mom_vol_z_90', 'ire_90'],
      dtype='object')

### 0.5. Info de dataset MNQ


In [131]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [132]:
info_dataset(mnq_alpha_factors)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 601
	Hora diaria de inicio 06:00
	Hora diaria de final 16:00
	Zona horaria: America/New_York


In [133]:
info_dataset(mnq_technical_indicators)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 601
	Hora diaria de inicio 06:00
	Hora diaria de final 16:00
	Zona horaria: America/New_York


## 1. Alineación de datasets

Empezaremos alineando horizontalmente los datasets `mnq_technical_indicators` y `mnq_alpha_factors`.

In [134]:
# Alineación horizontal por índice (datetime)
mnq_features_combined = pd.concat([mnq_technical_indicators, mnq_alpha_factors], axis=1)
# Eliminar columnas duplicadas (conservando la primera aparición)
mnq_features_combined = mnq_features_combined.loc[:, ~mnq_features_combined.columns.duplicated()]
# Confirmar que los índices estén alineados
mnq_features_combined = mnq_features_combined.sort_index()

In [135]:
mnq_features_combined.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'target_return_90',
       'rsi_14', 'rsi_7', 'momentum_10', 'momentum_5', 'macd', 'price_ema20',
       'price_ema30', 'stoch_k_20', 'stoch_k_30', 'bb_20', 'bb_30', 'bb_60',
       'atr_norm', 'roc_20', 'roc_30', 'roc_60', 'rev_mom_z_90',
       'rev_score_90', 'rev_mom_vol_z_90', 'ire_90'],
      dtype='object')

Luego alianeamos verticalmente los datasets `ic_technical_indicators` y `ic_alpha_factors`.


In [136]:
ic_features_combined = pd.concat([ic_technical_indicators, ic_alpha_factors], axis=0)

In [137]:
ic_features_combined = ic_features_combined.rename(columns={"indicador": "feature"})


In [138]:
ic_features_combined

,feature,ic_mean_90min,ic_std_90min
0,rsi_14,-0.126968,0.182209
1,rsi_7,-0.093018,0.136012
0,momentum_10,-0.081003,0.121896
1,momentum_5,-0.060196,0.086822
0,macd,-0.024934,0.102736
1,price_ema20,-0.109033,0.161206
2,price_ema30,-0.129262,0.187014
1,stoch_k_20,-0.098342,0.143963
2,stoch_k_30,-0.120082,0.172995
1,bb_20,-0.092459,0.138280


## 2. Criterio IC Score


Nuestro enfoque para la selección de los mejores features es el **IC Score**

El IC score combina dos métricas clave para evaluar la calidad de un indicador técnicos:

- |IC medio|: mide el poder predictivo promedio del indicador (cuánto se asocia su valor con el retorno futuro).

- IC std: mide la volatilidad o inestabilidad del indicador a lo largo del tiempo (días en tu caso).

El cociente `|IC| / std` representa un signal-to-noise ratio: cuánta señal útil aporta el indicador, ajustada por su variabilidad diaria.

Esto nos permite seleccionar features que no solo tienen buen rendimiento promedio, sino que además son consistentes. En otras palabras, es una forma de medir la **relación señal/ruido** de cada feature.

**Ventajas del IC Score como criterio de selección**

| Criterio | Descripción |
| --- | --- |
| **Objetividad** | Es una métrica cuantitativa clara, sin intervención manual ni subjetiva. |
| **Consistencia** | Premia indicadores que funcionan bien y consistentemente a lo largo de días. |
| **Robustez estadística** | Penaliza indicadores que parecen buenos pero son muy inestables (overfitting). |
| **Comparabilidad** | Permite ordenar decenas de indicadores heterogéneos en una sola escala. |
| **Modelado automatizado** | Compatible con pipelines de selección automática, sin intervención humana. |


**¿Por qué es mejor que usar solo el IC medio?**

Porque un factor puede tener un IC alto pero extremadamente volátil, lo cual indica que su utilidad es espuria o inconstante. En cambio, un factor con IC moderado pero muy estable suele ser más confiable para modelos reales.

En el trading cuantitativo institucional, es común que los equipos usen IC Score o variantes del Information Ratio (IR) como filtro previo antes de usar un factor en producción.

Ejemplo concreto:

- `atr_norm`: tiene el IC medio más alto (+0.0899), pero su std también es muy alta (0.2389) → bajo IC score ≈ 0.38

- `rsi_14`: IC medio menor (-0.0882), pero más estable (std = 0.17) → mejor IC score ≈ 0.51

Entonces, si tenés que elegir entre ellos para un modelo robusto, el IC score sugiere que `rsi_14` es más confiable, aunque su IC medio sea menor.

**Conclusión**

El IC score es una métrica más equilibrada, porque considera tanto la fuerza como la confiabilidad del factor.

Usarlo como criterio principal me protege de elegir factores ruidosos, y mejora la calidad de las señales que entrarán en mi modelo, especialmente si:

- Estoy evaluando muchos indicadores (como es nuestro caso).

- Voy a usar modelos complejos que son sensibles al ruido (como redes neuronales).

- En una etapa exploratoria podríamos automatizar y escalar el proceso de feature selection.

### 2.1. Calculo de IC Score para las ventanas de 30, 60 y 90 minutos.

Calcular la nueva columna IC_score para cada ventana de predicción

In [139]:
targets = ['90']
for target in targets:
  ic_features_combined[f"ic_score_{target}min"] =ic_features_combined[f"ic_mean_{target}min"].abs() / ic_features_combined[f"ic_std_{target}min"]


Con el siguiente código se generan tres datasets distintos, cada uno ordenado por el valor de IC_score correspondiente a su horizonte de predicción (30, 60 y 90 minutos).  De esta manera, es posible identificar qué indicadores técnicos tienen mayor peso relativo en cada ventana temporal.

In [140]:
ic_features_90 = ic_features_combined.sort_values(
    by="ic_score_90min", ascending=False
).reset_index(drop=True)

El siguiente código construye un nuevo dataset llamado `ic_features_comparacion`, donde se alinean horizontalmente las columnas `feature` de cada uno de los tres rankings generados previamente (`ic_features_30`, `ic_features_60` e `ic_features_90`).  

Cada columna contiene la lista ordenada de indicadores técnicos según su ic_score en el horizonte correspondiente (30, 60 y 90 minutos), lo que nos permite comparar fácilmente cuáles factores se mantienen, suben o bajan en importancia entre nuestras diferentes ventanas de predicción.

In [141]:
ic_features_comparison = pd.DataFrame({
    "features_to_90min": ic_features_90["feature"].reset_index(drop=True),
    "ic_score_90min": ic_features_90["ic_score_90min"].reset_index(drop=True),

})

### 2.2. Resultados

In [142]:
#Resultados basados en IC_score
ic_features_comparison

,features_to_90min,ic_score_90min
0,ire_90,2.489757
1,rev_mom_z_90,0.981206
2,rev_mom_vol_z_90,0.804866
3,roc_60,0.788505
4,bb_60,0.702669
5,rsi_14,0.696823
6,stoch_k_30,0.694134
7,momentum_5,0.693329
8,price_ema30,0.691188
9,rsi_7,0.683894


### 2.3. Análisis de factores por horizonte de predicción


1. Factores dominantes y estables

    - `ire_60` e `ire_90`  lideran claramente en todos los horizontes, con `ic_score` muy altos (>2). Son los factores más robustos para explicar retornos, independientemente de la ventana.  
    - `rev_mom_z_90` y `roc_60` se mantienen siempre en el top 5 de importancia. Tienen estabilidad y buena capacidad predictiva.  
    - `atr_norm` y `macd` aunque aparecen al final de los rankings, son consistentes: siempre están incluidos, lo que indica una señal débil pero persistente.  

2. Factores con estabilidad media

    - `bb_60` fuerte en 60 y 90 minutos, algo más abajo en 30 min. Su importancia crece con el horizonte.  
    - `stoch_k_30`, `stoch_k_20`, `price_ema30`, `price_ema20`, `rsi_14`, `rsi_7` siempre están presentes en posiciones intermedias, mostrando que aportan valor, aunque no son dominantes.  
    - `rev_score_60` y `rev_score_90` mantienen relevancia en los tres horizontes.  

3. Factores que ganan importancia en horizontes largos

    - `rev_mom_vol_z_90` es fuerte en 30 min (#5) y 60 min (#7), pero pierde peso en 90 min (#26). Podría ser un factor de corto/medio plazo.  
    - `rev_mom_z_45` y `rev_mom_z_60` su ranking mejora en 60 y 90 min, sugiriendo que capturan mejor reversión/momentum en ventanas largas.  
    - `rev_mom_z_90` claramente el más fuerte entre los reversales a plazos largos.  

4. Factores que tienden a rotar o perder fuerza

    - `momentum_5` y `momentum_10` son bajos en todos los horizontes, aunque siempre aparecen. Aportan señal, pero débil.  
    - `roc_20`, `roc_30` están mejor posicionados en 30 min, luego decaen en 90 min. Son más útiles en ventanas cortas.  
    - `rev_mom_z_30`  pierde relevancia al aumentar la ventana.  

5. Patrones generales

    - Los feature de reversión (`rev_mom_z`, `rev_score`, `rev_mom_vol_z`) ganan fuerza a medida que el horizonte crece. Ejemplo: `rev_mom_z_90` es top-3 siempre.  
    - Momentum clásico (`roc`, `momentum`, `ema`, `rsi`, `stoch`, `bb`) aportan pero se ven desplazados en horizontes largos por factores de reversión.  
    - Volatilidad (`atr_norm`) no es dominante, pero constante: puede ser útil como complemento en modelos, aunque no como feature principal.  

### 2.4. Conclusión basada en análisis de ic_score

- Factores robustos y no negociables: `ire_60`, `ire_90`, `rev_mom_z_90`, `roc_60`, `bb_60`.  
- Factores secundarios que añaden diversificación: `stoch_k_30`, `stoch_k_20`, `price_ema30`, `rsi_14`, `rev_score_90`.  
- Factores complementarios: `atr_norm`, `macd`, `momentum_5`, `momentum_10`.  

## 3. Correlación

Ya ordenamos los indicadores por IC Score, que es un métrica robusta de señal ajustada por estabilidad. Pero necesitamos controlar la redundancia (multicolinealidad). Ya que indicadores con alta correlación podrían aportar información duplicada, y:

- Distorsionar modelos lineales
- Inflar varianzas en los coeficientes
- Hacer innecesariamente complejo el modelo

**¿Cómo lo controlo?**
<br>
Usaré una matriz de correlación entre indicadores, típicamente sobre sus valores normalizados (z-score por día) para detectar grupos redundantes.


#### 3.1. Construcción de dataset de features

Recordemos que tenemos un listado con los features a conservar, filtramos el dataset para que solo queden las columnas correspondientes a esos features:

In [143]:
columnas_base = ['date', 'open', 'high', 'low','close','volume']
columnas_target = ['target_return_90']

# Unimos las listas de columnas a eliminar
cols_drop = columnas_base + columnas_target

# Eliminamos esas columnas
mnq_features = mnq_features_combined.drop(columns=cols_drop, errors="ignore")

### 3.2. Calculo de correlación

Calculamos la matriz de correlación:

In [144]:
mnq_features_corr = mnq_features.corr()

### 3.3. Buscamos la mayor correlación de cada feature

Teniendo:
- ic_features_combined: dataset con columna 'feature'
- mnq_features: dataset con todos los features calculados
- mnq_features_corr : matriz de correlación

Vamos a calcular los 3 factores más correlacionados con cada factor:

In [145]:
# Para cada factor en Iic_features_combined, encontrar los 5 más correlacionados
top5_features = []
top5_valores = []

for feature in ic_features_combined["feature"]:
    if feature in mnq_features_corr.columns:
        corrs = mnq_features_corr[feature].drop(labels=[feature])  # eliminar autocorrelación
        top5 = corrs.abs().sort_values(ascending=False).head(5)
        features = top5.index.tolist()
        valores = [mnq_features_corr.loc[feature, f] for f in features]
    else:
        features = [None, None, None]
        valores = [None, None, None]

    top5_features.append(features)
    top5_valores.append(valores)

# Paso 4: añadir al DataFrame original como columnas separadas
ic_features_combined["corr_ind_1"] = [f[0] for f in top5_features]
ic_features_combined["corr_value_1"] = [v[0] for v in top5_valores]

ic_features_combined["corr_ind_2"] = [f[1] for f in top5_features]
ic_features_combined["corr_value_2"] = [v[1] for v in top5_valores]

ic_features_combined["corr_ind_3"] = [f[2] for f in top5_features]
ic_features_combined["corr_value_3"] = [v[2] for v in top5_valores]


In [146]:
# Seleccionar solo las columnas deseadas
ic_top_corr= ic_features_combined[["feature", "corr_ind_1", "corr_value_1", "corr_ind_2", "corr_value_2", "corr_ind_3", "corr_value_3",]].copy()

### 3.4. Resultados

In [147]:
ic_top_corr

,feature,corr_ind_1,corr_value_1,corr_ind_2,corr_value_2,corr_ind_3,corr_value_3
0,rsi_14,bb_30,0.936817,rsi_7,0.935937,stoch_k_30,0.925667
1,rsi_7,bb_20,0.945782,rsi_14,0.935937,stoch_k_20,0.920604
0,momentum_10,price_ema20,0.895628,price_ema30,0.847076,macd,0.824091
1,momentum_5,price_ema20,0.790955,momentum_10,0.707255,price_ema30,0.700584
0,macd,momentum_10,0.824091,price_ema20,0.710952,momentum_5,0.666800
1,price_ema20,price_ema30,0.980547,momentum_10,0.895628,roc_20,0.864440
2,price_ema30,price_ema20,0.980547,roc_20,0.902281,roc_30,0.865726
1,stoch_k_20,bb_20,0.948147,stoch_k_30,0.927894,bb_30,0.921815
2,stoch_k_30,bb_30,0.947384,stoch_k_20,0.927894,rsi_14,0.925667
1,bb_20,stoch_k_20,0.948147,rsi_7,0.945782,bb_30,0.941943


### 3.5. Análisis de resultados

1. Indicadores clásicos de momentum y osciladores

    - `rsi_14` está fuertemente correlacionado con `rsi_7`, `bb_30` y `stoch_k_30`.  
    - `rsi_7` se conecta con `bb_20`, `rsi_14`, `stoch_k_20`.  

    Esto confirma que los RSI están muy ligados a osciladores tipo stochastics y bandas de Bollinger. Si los uso juntos puedo redundar en información repetida.
<br>

2. Momentum lineal y promedios móviles

    - `momentum_10` está cercano a `price_ema20`, `price_ema30`, `macd`.  
    - `momentum_5` está ligado a `price_ema20`, `momentum_10`, `price_ema30`.  
    - `macd` está conectado a `momentum_10`, `price_ema20`, `momentum_5`.  
    - `price_ema20` y `price_ema30` están muy correlacionados entre sí y también con `momentum` y `roc`.  

    Esto dibuja un clúster de momentum tendencial, donde todos capturan la misma señal de continuidad de tendencia. Si incluyo demasiados de este grupo me pueden generar colinealidad.
<br>

3. Osciladores estocásticos y Bollinger

    - `stoch_k_20` ligado a `bb_20`, `stoch_k_30`, `bb_30`.  
    - `stoch_k_30` ligado a `bb_30`, `stoch_k_20`, `rsi_14`.  
    - `bb_20` ligado a `stoch_k_20`, `rsi_7`, `bb_30`.  
    - `bb_30` ligado a `rev_score_30`, `stoch_k_30`, `bb_20`.  
    - `bb_60` ligado a `rev_score_60`, `rev_score_45`, `rsi_14`.  

    Aquí se ve un clúster claro de osciladores-volatilidad. Están fuertemente interrelacionados y tienden a medir sobrecompra/sobreventa y rangos de precios. El riesgo de redundancia es alto.
<br>

4. Factores de reversión de precio

    - Los `rev_mom_z_*` se relacionan entre sí y también con `rev_score` y `rev_mom_vol_z`.  
      - Por ejemplo el `rev_mom_z_90` está correlacionado con `rev_mom_vol_z_90`, `rev_mom_z_60`, `rev_score_90`.  
    - Los `rev_score_*` forman un subgrupo muy cohesionado, relacionados con las Bollinger (`bb_30`, `bb_60`).  
    - Las `rev_mom_vol_z_*` están interconectados entre sí y con `rev_mom_z`.  

    Esto indica que la familia de reversal scores y momentum de reversión es muy densa en correlaciones internas, casi redundante. Probablemente baste con elegir un par representativos (como por ej. `rev_mom_z_90`, `rev_score_90`, `rev_mom_vol_z_60`).
<br>

5. Retornos intradía extremos (IRE)

    - `ire_60` ligado a `ire_90`, `rev_mom_z_90`, `rev_score_90`.  
    - `ire_90` ligado a `ire_60`, `rev_mom_z_90`, `rev_score_90`.  

    Este par es extremadamente fuerte y además conecta con los reversales. Los IRE aportan una visión macro de agotamiento intradía que está alineada con reversión.
<br>

6. Volatilidad

    - El `atr_norm` está correlacionado con `ire_90`, `ire_60`, `bb_60`.  

    El ATR queda como un puente entre volatilidad clásica y factores de reversión intradía.


### 3.6. Conclusiones de correlación

1. Hay clústers muy definidos:  
    - Momentum y medias móviles (`momentum`, `ema`, `macd`, `roc`).  
    - Osciladores y volatilidad (`rsi`, `stoch_k`, `bb`).  
    - Reversión (`rev_mom_z`, `rev_score`, `rev_mom_vol_z`).  
    - IRE (`ire_60`, `ire_90`).  

2. Dentro de cada clúster, los factores son altamente correlacionados, por lo que elegir demasiados genera redundancia.  

3. Una buena estrategia sería seleccionar 1 o 2 factores representativos por clúster:  
    - Momentum: `roc_60` y `price_ema30`.  
    - Osciladores: `bb_60` y `stoch_k_30`.  
    - Reversión: `rev_mom_z_90` y `rev_score_90`.  
    - IRE: `ire_60` o `ire_90`.  
    - Volatilidad: `atr_norm`.  

De esta forma, nuestro modelo tendría cobertura balanceada sin sobrecargarlo con señales repetitivas.

## 4. Análisis en conjunto

In [148]:
#ic_top_corr

In [149]:
#ic_features_comparison

### 4.1. Estabilidad de features entre horizontes

Evaluar qué tan estables son los features entre los distintos horizontes de predicción (30, 60 y 90 min).

Para eso:

- Tomamos solo el Top-N de cada horizonte (por defecto N=10).
- Contamos en cuántos horizontes aparece cada feature.
- Calculamos el promedio de ranking (avg_rank) solo en los horizontes donde aparece.
- Marcamos como ESTABLE a cualquier feature que aparezca en 2 o más horizontes.
- Ordenamos la tabla resultante por:
  - presencia_en_horizontes (descendente),
  - avg_rank (ascendente).

#### 4.1.1. Código

In [150]:
def _dedup_preserving_order(seq):
    """Elimina duplicados de una lista/serie preservando el orden original."""
    seen = set()
    result = []
    for item in seq:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

In [151]:
def compute_stability_from_ic_features_comparison(
    ic_features_comparison: pd.DataFrame,
    top_n: int = 10
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    PASO 3 directo desde ic_features_comparison (ancho).
    Devuelve:
      - stable_features: tabla de estabilidad (feature, presencia_en_horizontes, avg_rank, estable)
      - rank_df: ranking largo generado internamente (rank, feature, horizon)
    """
    # 1) Verificación de columnas esperadas
    #required = ["features_to_30min", "features_to_60min", "features_to_90min"]
    required = ["features_to_90min"]
    missing = [c for c in required if c not in ic_features_comparison.columns]
    if missing:
        raise ValueError(f"Faltan columnas: {missing}")

    # 2) Limpieza + deduplicación por columna preservando mejor ranking
    cleaned_cols = {}
    max_len = 0
    for col in required:
        col_clean = _dedup_preserving_order(ic_features_comparison[col])
        cleaned_cols[col] = col_clean
        max_len = max(max_len, len(col_clean))

    # 3) Reconstrucción (alineada por filas, con NaN si hace falta) y ranking largo
    df_clean = pd.DataFrame({
        col: pd.Series(vals + [pd.NA]*(max_len - len(vals)))
        for col, vals in cleaned_cols.items()
    })

    rows = []
    for col in required:
        horizon = col.split("_to_")[1].replace("min", "")  # '30','60','90'
        series = df_clean[col].dropna().astype(str)
        for idx, feat in enumerate(series):
            rows.append({"rank": idx + 1, "feature": feat, "horizon": horizon})
    rank_df = pd.DataFrame(rows, columns=["rank","feature","horizon"])

    # 4) Estabilidad en Top-N
    top = rank_df[rank_df["rank"] <= top_n].copy()
    presence = top.groupby("feature")["horizon"].nunique().rename("presencia_en_horizontes")
    avg_rank = top.groupby("feature")["rank"].mean().rename("avg_rank")

    stable_features = (
        pd.concat([presence, avg_rank], axis=1)
          .reset_index()
          .assign(estable=lambda d: d["presencia_en_horizontes"] >= 2)
          .sort_values(["presencia_en_horizontes", "avg_rank"], ascending=[False, True])
          .reset_index(drop=True)
    )

    return stable_features, rank_df

In [152]:
stable_features, rank_df = compute_stability_from_ic_features_comparison(ic_features_comparison, top_n=15)

#### 4.1.2. Resultados

In [153]:
stable_features

,feature,presencia_en_horizontes,avg_rank,estable
0,ire_90,1,1.0,False
1,rev_mom_z_90,1,2.0,False
2,rev_mom_vol_z_90,1,3.0,False
3,roc_60,1,4.0,False
4,bb_60,1,5.0,False
5,rsi_14,1,6.0,False
6,stoch_k_30,1,7.0,False
7,momentum_5,1,8.0,False
8,price_ema30,1,9.0,False
9,rsi_7,1,10.0,False


- `feature`: el indicador o factor evaluado  
- `presencia_en_horizontes`: en cuántos horizontes (30, 60, 90) aparece dentro del Top-N elegido  
- `avg_rank`: el promedio del ranking en esos horizontes, cuanto más bajo mejor  
- `estable`: True si aparece en dos o más horizontes, False si aparece en solo uno  

#### 4.1.3. Interpretación

1. Features muy estables y fuertes (presentes en los 3 horizontes, con buen ranking)  
   - ire_60 (presente en los 3, rank medio ≈ 1.3)  
   - ire_90 (presente en los 3, rank medio ≈ 1.7)  
   - rev_mom_z_90, roc_60, rev_mom_z_60 (todos con rank medio ≤ 5)  
   Son candidatos principales multihorizonte, constantes y siempre bien posicionados.  

2. Features estables en los 3 horizontes pero un poco más abajo en el ranking  
   - bb_60 (avg rank 7), rev_score_90 (8), rev_mom_z_45 (9.3), stoch_k_30 (9.7), rsi_14 (12), price_ema30 (12.3)  
   También son estables, aunque no tan dominantes como el grupo anterior. Funcionan como candidatos secundarios o representantes de clusters correlacionados.  

3. Features estables pero solo en 2 horizontes  
   - rev_mom_vol_z_90, rev_mom_vol_z_60, stoch_k_20, bb_30  
   Interesantes pero menos consistentes. Posibles candidatos complementarios.  

4. Features inestables (solo en 1 horizonte, estable=False)  
   - price_ema20, rev_score_60, rev_mom_vol_z_45, roc_30  
   Aparecen aislados en un único horizonte, menos confiables. Se usarían solo para explorar señales muy específicas.  


#### 4.1.4. Conclusiones parciales

- Top tier estables y fuertes (multihorizonte, avg_rank bajo):  
  ire_60, ire_90, rev_mom_z_90, roc_60, rev_mom_z_60  

- Estables secundarios (multihorizonte, avg_rank medio):  
  bb_60, rev_score_90, rev_mom_z_45, stoch_k_30, rsi_14, price_ema30  

- Estables parciales (2 horizontes):  
  rev_mom_vol_z_90, rev_mom_vol_z_60, stoch_k_20, bb_30  

- Inestables (1 horizonte, descartables para shortlist robusto):  
  price_ema20, rev_score_60, rev_mom_vol_z_45, roc_30  

### 4.2. Análisis de clúster por correlación

Objetivo:

- Construir un grafo no dirigido donde:
    - Cada feature es un nodo.
    - Se conecta con sus corr_ind_1, corr_ind_2, corr_ind_3 desde ic_top_corr_clean.
- Luego buscamos las componentes conexas del grafo (= clusters de features muy correlacionados).
- Para cada cluster y cada horizonte, se elegirá un principal: el feature con mejor rank en ese horizonte.
- El resto serán sus sustitutos.



#### 4.2.1. Código

In [154]:
from collections import defaultdict
import numpy as np
import pandas as pd

def build_clusters_with_threshold_from_ic_norank(
    ic_top_corr_clean: pd.DataFrame,
    ic_features_comparison: pd.DataFrame,
    corr_matrix: pd.DataFrame | None = None,
    threshold: float = 0.8
) -> tuple[pd.DataFrame, dict]:
    """
    PASO 4 con umbral, tomando ic_features_comparison directamente (sin construir rank_df).
    - Para cada cluster y horizonte, el principal es el feature del cluster que aparezca
      primero en la lista de ic_features_comparison[col].
    - El resto de los features del cluster se consideran sustitutos.
    """
    # --- 0) Conjuntos base de nodos ---
    nodes_from_ic = set()
    #for col in ["features_to_30min","features_to_60min","features_to_90min"]:
    for col in ["features_to_90min"]:
        nodes_from_ic |= set(ic_features_comparison[col].dropna().astype(str).tolist())
    nodes_from_iccorr = set(ic_top_corr_clean["feature"].astype(str).tolist())
    for i in (1,2,3):
        if f"corr_ind_{i}" in ic_top_corr_clean.columns:
            nodes_from_iccorr |= set(ic_top_corr_clean[f"corr_ind_{i}"].astype(str).tolist())
    all_nodes = {n.strip() for n in (nodes_from_ic | nodes_from_iccorr) if isinstance(n,str) and n.strip()}

    # --- 1) Inicializar grafo ---
    graph = defaultdict(set)
    for n in all_nodes:
        graph[n]

    edges_added = 0
    mode_used = None
    warned_no_values = False

    # --- 2) Construcción de aristas ---
    if corr_matrix is not None:
        mode_used = "corr_matrix"
        avail = list(all_nodes & set(corr_matrix.index) & set(corr_matrix.columns))
        sub = corr_matrix.loc[avail,avail].copy().astype(float)
        sub.values[~np.isfinite(sub.values)] = 0.0
        for i in range(len(avail)):
            fi = avail[i]
            rho = sub.iloc[i,(i+1):].values
            partners = avail[(i+1):]
            mask = np.abs(rho) >= threshold
            for g in np.array(partners)[mask]:
                if g!=fi:
                    graph[fi].add(g)
                    graph[g].add(fi)
                    edges_added += 1
    else:
        mode_used = "ic_top_corr"
        has_vals = all(col in ic_top_corr_clean.columns for col in ["corr_val_1","corr_val_2","corr_val_3"])
        for _,row in ic_top_corr_clean.iterrows():
            f = str(row["feature"]).strip()
            if not f: continue
            for i in (1,2,3):
                g = str(row.get(f"corr_ind_{i}","")).strip()
                if not g: continue
                use_edge = True
                if has_vals:
                    try:
                        val = float(row.get(f"corr_val_{i}",np.nan))
                        use_edge = np.isfinite(val) and (abs(val)>=threshold)
                    except:
                        use_edge = False
                else:
                    if not warned_no_values:
                        print("[Aviso] ic_top_corr_clean no trae corr_val_1..3 → no se puede aplicar threshold real. "
                              "Se conectarán los 3 vecinos top por fila (posible mega-cluster).")
                        warned_no_values = True
                if use_edge:
                    graph[f].add(g)
                    graph[g].add(f)
                    edges_added += 1

    # --- 3) Componentes conexas ---
    visited=set()
    clusters=[]
    for node in graph:
        if node not in visited:
            stack=[node]; comp=[]
            while stack:
                cur=stack.pop()
                if cur not in visited:
                    visited.add(cur); comp.append(cur)
                    stack.extend(graph[cur]-visited)
            clusters.append(comp)

    # --- 4) Determinar principal/sustitutos directamente con ic_features_comparison ---
    cluster_info=[]
    for cid,comp in enumerate(clusters,start=1):
        entry={"cluster_id":cid,"features":comp}
        comp_set=set(comp)
        #for horizon,col in zip(["30","60","90"],["features_to_30min","features_to_60min","features_to_90min"]):
        for horizon,col in zip(["90"],["features_to_90min"]):
            series=ic_features_comparison[col].dropna().astype(str).tolist()
            # primer feature del cluster que aparezca en esa columna
            principal=None
            for feat in series:
                if feat in comp_set:
                    principal=feat; break
            if principal:
                entry[f"principal_{horizon}"]=principal
                entry[f"sustitutos_{horizon}"]=[f for f in comp if f!=principal]
            else:
                entry[f"principal_{horizon}"]=None
                entry[f"sustitutos_{horizon}"]=[]
        cluster_info.append(entry)

    cluster_df=pd.DataFrame(cluster_info)
    report={
        "mode":mode_used,
        "threshold":threshold,
        "n_nodes":len(all_nodes),
        "n_edges":edges_added,
        "n_clusters":len(clusters),
        "note":("Sin corr_val_1..3: no se aplicó umbral real; se usaron 3 vecinos top por fila."
                if (mode_used=="ic_top_corr" and not has_vals) else "")
    }
    return cluster_df, report

In [155]:
feature_clusters, clusters_report = build_clusters_with_threshold_from_ic_norank(
     ic_top_corr, ic_features_comparison, corr_matrix=mnq_features_corr, threshold=0.85
 )

#### 4.2.2. Resultados

In [156]:
feature_clusters

,cluster_id,features,principal_90,sustitutos_90
0,1,[rev_mom_z_90],rev_mom_z_90,[]
1,2,"[rsi_7, bb_20, stoch_k_20, rsi_14, bb_30, bb_6...",bb_60,"[rsi_7, bb_20, stoch_k_20, rsi_14, bb_30, rev_..."
2,3,"[roc_30, price_ema30, roc_20, price_ema20, mom...",price_ema30,"[roc_30, roc_20, price_ema20, momentum_10]"
3,4,[atr_norm],atr_norm,[]
4,5,[macd],macd,[]
5,6,[rev_mom_vol_z_90],rev_mom_vol_z_90,[]
6,7,[ire_90],ire_90,[]
7,8,[roc_60],roc_60,[]
8,9,[momentum_5],momentum_5,[]


In [157]:
clusters_report

{'mode': 'corr_matrix',
 'threshold': 0.85,
 'n_nodes': 20,
 'n_edges': 24,
 'n_clusters': 9,
 'note': ''}

#### 4.2.3. Interpretación de los Clusters (umbral 0.85)

1. **Cluster 1: Bollinger + Reversal Score**
   - **Features**: `bb_20, bb_30, bb_60, rev_score_30, rev_score_45, rev_score_60, rev_score_90`
   - **Principales**:
     - 30m → `rev_score_90`
     - 60m → `bb_60`
     - 90m → `bb_60`
   - **Interpretación**: Este cluster agrupa indicadores de volatilidad y reversal (`bb_*`, `rev_score_*`).  
     - Para horizontes largos domina **`bb_60`**.  
     - Para horizontes cortos (30m) aparece **`rev_score_90`** como más fuerte.  
     - Los demás (`bb_20`, `bb_30`, `rev_score_30/45/60`) se consideran **sustitutos**.

2. **Cluster 2: Indicadores de rango inicial (IRE)**
   - **Features**: `ire_60, ire_90`
   - **Principales**:
     - 30m → `ire_90`
     - 60m → `ire_60`
     - 90m → `ire_60`
   - **Interpretación**: Son muy similares entre sí.  
     - Para horizontes más cortos domina **`ire_90`**, mientras que en horizontes más largos gana **`ire_60`**.

3. **Cluster 3: Reversal Momentum Volatility**
   - **Features**: `rev_mom_vol_z_45, rev_mom_vol_z_60`
   - **Principales**:
     - 30m y 60m → `rev_mom_vol_z_60`
     - 90m → `rev_mom_vol_z_45`
   - **Interpretación**: Factores muy relacionados; el rol de “principal” se alterna según el horizonte.

4. **Cluster 4: Momentum ROC**
   - **Features**: `roc_60`
   - **Interpretación**: Es un **cluster unitario**. Siempre se elige `roc_60` como principal.

5. **Cluster 5: Reversal Momentum Volatility (90)**
   - **Features**: `rev_mom_vol_z_90`
   - **Interpretación**: Otro **cluster unitario**.

6. **Cluster 6: Reversal Momentum**
   - **Features**: `rev_mom_z_90`
   - **Interpretación**: Otro **cluster unitario**.

7. **Cluster 7: EMA + Momentum + ROC**
   - **Features**: `momentum_10, price_ema20, price_ema30, roc_20, roc_30`
   - **Principal**: `price_ema30` en todos los horizontes.
   - **Interpretación**: Este cluster mezcla momentum clásico y medias móviles.  
     - **`price_ema30`** es claramente el más fuerte y consistente.  
     - Los demás se consideran sustitutos (`momentum_10`, `roc_20`, `roc_30`, `price_ema20`).

8. **Cluster 8: Reversal Momentum (45)**
   - **Features**: `rev_mom_z_45`
   - **Interpretación**: Cluster unitario.

9. **Cluster 9: MACD**
   - **Features**: `macd`
   - **Interpretación**: Cluster unitario.

10. **Cluster 10: Reversal Momentum (30)**
    - **Features**: `rev_mom_z_30`
    - **Interpretación**: Cluster unitario.

11. **Cluster 11: Momentum corto**
    - **Features**: `momentum_5`
    - **Interpretación**: Cluster unitario.

12. **Cluster 12: ATR Normalizado**
    - **Features**: `atr_norm`
    - **Interpretación**: Cluster unitario.

13. **Cluster 13: Reversal Momentum (60)**
    - **Features**: `rev_mom_z_60`
    - **Interpretación**: Cluster unitario.




#### 4.4.3. Conclusiones


- El umbral 0.85 permitió separar en **13 clusters distintos**.  
- Cluster 1 es el más grande, mezcla Bollinger, reversal score y osciladores, donde bb_60 y rev_score_90 son los líderes.  
- Clusters 2 y 3 agrupan pares/familias pequeñas con alternancia de principal según el horizonte.  
- Clusters 4 al 13 son unitarios, siempre con un único representante.  
- Esto simplifica el paso siguiente: al construir shortlists, basta elegir un representante por cluster, reduciendo redundancia.


### 4.3. Generación de shortlist NO REDUNDANTE por horizonte (greedy)

Ahora intentaremos construir, para cada horizonte (30/60/90 minutos), una lista “shortlist” de factores sin redundancias fuertes.

Partiendo del ranking dado por `ic_features_comparison` y evitaremos incluir dos factores muy correlacionados entre sí, aun cuando pertenezcan a clusters distintos. Si un cluster no puede aportar un factor que cumpla el umbral de correlación, descartaremos ese cluster.

Entradas

  - `ic_features_comparison`: DataFrame ancho con tres columnas ordenadas por importancia: features_to_30min, features_to_60min, features_to_90min. El orden de filas es el ranking (fila 0 es mejor).

  - `feature_clusters`: DataFrame con clusters de redundancia (cada fila: cluster_id, features, y el “principal” por horizonte más sus “sustitutos`).

  - `corr_matrix`: matriz de correlaciones (índice y columnas = nombres de features), simétrica, por defecto `mnq_features_corr`.

  - `k`: tamaño objetivo del shortlist aceptado por horizonte.

  - `corr_threshold`: umbral máximo de correlación absoluta permitido entre el candidato y cualquier feature ya aceptado.

Salida

  - Un diccionario con tres DataFrames: `recommended_30`, `recommended_60`, `recommended_90`.

#### 4.3.1. Código

In [158]:
def build_shortlists_with_intercluster_corrfilter_and_discards(
    ic_features_comparison: pd.DataFrame,
    feature_clusters: pd.DataFrame,
    corr_matrix: pd.DataFrame,
    k: int = 12,
    corr_threshold: float = 0.7,
) -> dict[str, pd.DataFrame]:
    """
    Greedy por horizonte + filtro estricto inter-clusters + registro de descartes.
    Devuelve: {'recommended_30': df, 'recommended_60': df, 'recommended_90': df}
    Cada df incluye columnas: rank, feature, cluster_id, rol, motivo_de_inclusion, ic_score
    """

    # --- Mapeos auxiliares ---
    # feature -> cluster_id
    feat2cluster = {}
    for _, row in feature_clusters.iterrows():
        cid = int(row["cluster_id"])
        for f in row["features"]:
            feat2cluster[str(f)] = cid

    # cluster_id -> row (para acceder a principal/sustitutos por horizonte)
    clusters_by_id = {int(r["cluster_id"]): r for _, r in feature_clusters.iterrows()}

    # ranking (feature -> rank) por horizonte, para ordenar sustitutos por mejor posición
    rank_index = {}
    #for horizon, col in zip(["30", "60", "90"],["features_to_30min", "features_to_60min", "features_to_90min"]):
    for horizon, col in zip(["90"],["features_to_90min"]):
        series = ic_features_comparison[col].dropna().astype(str).tolist()
        rank_index[horizon] = {f: i+1 for i, f in enumerate(series)}  # 1-based

    def _principal_flag(feature: str, cid: int, horizon: str) -> bool:
        row = clusters_by_id.get(cid)
        if row is None:
            return True
        return (row.get(f"principal_{horizon}", None) == feature)

    def _corr_ok_with_all(candidate: str, accepted_feats: list[str], thr: float) -> bool:
        for g in accepted_feats:
            if candidate not in corr_matrix.index or g not in corr_matrix.columns:
                return False
            rho = corr_matrix.loc[candidate, g]
            if not np.isfinite(rho) or abs(rho) > thr:
                return False
        return True

    def _sorted_substitutes_by_rank(cid: int, horizon: str) -> list[str]:
        row = clusters_by_id.get(cid)
        if row is None:
            return []
        subs = list(row.get(f"sustitutos_{horizon}", []))
        ri = rank_index[horizon]
        subs.sort(key=lambda f: ri.get(f, 10**9))
        return subs

    results = {}
    for horizon, feat_col, score_col in zip(
        ["90"], #["30", "60", "90"],
        ["features_to_90min"],
        #["features_to_30min", "features_to_60min", "features_to_90min"],
        ["ic_score_90min"],
        #["ic_score_30min", "ic_score_60min", "ic_score_90min"],

        ):
        # --- Mapa feature -> ic_score del horizonte correspondiente ---
        # Si un feature aparece varias veces, nos quedamos con el primer ic_score (mejor rank).
        score_map = (
            ic_features_comparison[[feat_col, score_col]]
            .dropna(subset=[feat_col])
            .astype({feat_col: str})
            .drop_duplicates(subset=[feat_col], keep="first")
            .set_index(feat_col)[score_col]
            .to_dict()
        )

        used_clusters = set()
        rejected_clusters = set()
        accepted_rows = []
        discarded_rows = []
        accepted_feats = []
        series = ic_features_comparison[feat_col].dropna().astype(str).tolist()

        for rk, f in enumerate(series, start=1):
            if len(accepted_rows) >= k:
                break

            cid = feat2cluster.get(f)
            if cid is None:
                cid = -abs(hash(("orphan", f))) % (10**9)

            if cid in used_clusters or cid in rejected_clusters:
                continue

            is_principal = _principal_flag(f, cid, horizon)
            cand_feature = f
            cand_role = "principal" if is_principal else "sustituto"
            cand_motivo = (
                f"mejor del cluster {cid}" if is_principal
                else f"primero del cluster {cid} en el ranking (sustituto del principal)"
            )

            conflict = False
            if accepted_feats:
                if cand_feature in corr_matrix.index and all(g in corr_matrix.columns for g in accepted_feats):
                    max_rho = max(abs(corr_matrix.loc[cand_feature, g]) for g in accepted_feats)
                    conflict = not (np.isfinite(max_rho) and max_rho <= corr_threshold)
                else:
                    conflict = True  # conservador

            if conflict:
                subs = _sorted_substitutes_by_rank(cid, horizon)
                alt_found = None
                for alt in subs:
                    if _corr_ok_with_all(alt, accepted_feats, corr_threshold):
                        alt_found = alt
                        break

                if alt_found is None:
                    discarded_rows.append({
                        "rank": rk,
                        "feature": cand_feature,
                        "cluster_id": cid,
                        "rol": "descartado",
                        "motivo_de_inclusion": (
                            f"descartado por alta correlación (> {corr_threshold}) con features ya aceptados; "
                            f"sin sustitutos válidos en el cluster"
                        ),
                        "ic_score": score_map.get(cand_feature, np.nan),
                    })
                    rejected_clusters.add(cid)
                    continue

                cand_feature = alt_found
                cand_role = "sustituto"
                cand_motivo = f"reemplazo por alta correlación (> {corr_threshold}) con features ya aceptados"

            accepted_rows.append({
                "rank": rk,
                "feature": cand_feature,
                "cluster_id": cid,
                "rol": cand_role,
                "motivo_de_inclusion": cand_motivo,
                "ic_score": score_map.get(cand_feature, np.nan),
            })
            accepted_feats.append(cand_feature)
            used_clusters.add(cid)

        final_rows = accepted_rows + discarded_rows
        df_final = (
            pd.DataFrame(final_rows, columns=["rank","feature", "ic_score", "cluster_id","rol","motivo_de_inclusion",])
            .sort_values(
                ["rol","rank"],
                key=lambda s: s.map({"principal":0,"sustituto":1,"descartado":2}).fillna(3)
            )
            .reset_index(drop=True)
        )
        results[f"recommended_{horizon}"] = df_final

    return results

In [159]:
shortlists_strict_discards = build_shortlists_with_intercluster_corrfilter_and_discards(
    ic_features_comparison,
    feature_clusters,
    corr_matrix=mnq_features_corr,   # DataFrame simétrico index/columns=features
    k=12,
    corr_threshold=0.7
)

#recommended_30 = shortlists_strict_discards["recommended_30"]
#recommended_60 = shortlists_strict_discards["recommended_60"]
recommended_90 = shortlists_strict_discards["recommended_90"]

#### 4.3.2. Resultados y análisis

##### Para la ventana de 90min:

In [187]:
recommended_90

,rank,feature,ic_score,cluster_id,rol,motivo_de_inclusion
0,1,ire_90,2.489757,7,principal,mejor del cluster 7
1,2,rev_mom_z_90,0.981206,1,principal,mejor del cluster 1
2,4,roc_60,0.788505,8,principal,mejor del cluster 8
3,5,bb_60,0.702669,2,principal,mejor del cluster 2
4,8,momentum_5,0.693329,9,principal,mejor del cluster 9
5,19,atr_norm,0.283756,4,principal,mejor del cluster 4
6,20,macd,0.242702,5,principal,mejor del cluster 5
7,9,roc_20,0.658728,3,sustituto,reemplazo por alta correlación (> 0.7) con fea...
8,3,rev_mom_vol_z_90,0.804866,6,descartado,descartado por alta correlación (> 0.7) con fe...


1. Factores principales aceptados

    - Muy fuerte (core):
      - `ire_60` (IC_score = 2.48, cluster 9)

    - Moderados/aceptables:
      - `rev_mom_z_90` (0.94)
      - `roc_60` (0.93)
      - `bb_60` (0.74)
      - `momentum_5` (0.86)

    - Marginales:
      - `atr_norm` (0.32)
      - `macd` (0.29)
      - `roc_20` (0.85) [sustituto elegido por correlación]
      - `rev_mom_vol_z_60` (0.84) [sustituto elegido por correlación]

2. Clusters descartados

    Eliminados por alta correlación (>0.7) con features ya aceptados:

    - `rev_mom_z_60` (0.93)  
    - `rev_mom_z_45` (0.92)  
    - `rev_mom_z_30` (0.88)  
    - `rev_mom_vol_z_90` (0.83)  

    Se retuvieron como sustitutos `roc_20` y `rev_mom_vol_z_60`, lo que permitió mantener diversidad sin violar el umbral de correlación.

3. Balance del shortlist

      - Diversidad de clusters: se logró una buena representación con al menos un feature por cluster.  
      - Distribución de IC_score: un líder sólido (`ire_60`), varios factores fuertes entre 0.8 y 0.94, y algunos marginales con valores bajos.  
      - Riesgo: `atr_norm` y `macd` continúan mostrando baja relevancia estadística y podrían introducir ruido.

4. Recomendación práctica

    - Mantener como core: `ire_60`  
    - Mantener como secundarios: `rev_mom_z_90`, `roc_60`, `bb_60`, `momentum_5`  
    - Evaluar en validación: `roc_20`, `rev_mom_vol_z_60`, `atr_norm`, `macd`  
    - Clusters descartados: correcta eliminación de señales redundantes de reversión/momentum, priorizando diversidad.

#### 4.3.3. Listado de features por ventana de predicción


En los tres horizontes evaluados (30, 60 y 90 minutos), los features `atr_norm` y `macd` presentan valores de `IC_score` consistentemente bajos (menores a 0.4). Esto indica que su capacidad predictiva es débil y difícilmente distinguible del ruido.  

Además, ambos se mantienen como marginales en cada shortlist, sin mostrar mejoras relevantes ni aportar robustez estadística frente al resto de factores seleccionados.  

Por estas razones, se concluye que `atr_norm` y `macd` no aportan valor real al modelo y se decide eliminarlos del listado final de features en todas las ventanas de predicción.

In [188]:
#features_to_30 = recommended_30.loc[recommended_30["rol"].isin(["principal", "sustituto"]), "feature"].tolist()
#features_to_60 = recommended_60.loc[recommended_60["rol"].isin(["principal", "sustituto"]), "feature"].tolist()
features_to_90 = recommended_90.loc[recommended_90["rol"].isin(["principal", "sustituto"]), "feature"].tolist()

In [189]:
features_to_90

['ire_90',
 'rev_mom_z_90',
 'roc_60',
 'bb_60',
 'momentum_5',
 'atr_norm',
 'macd',
 np.str_('roc_20')]

In [207]:
# Filtro para remover 'macd' y 'atr_norm'
to_remove = {'macd', 'atr_norm', 'roc_20'}

#features_to_30 = [f for f in features_to_30 if f not in to_remove]
#features_to_60 = [f for f in features_to_60 if f not in to_remove]
features_to_90 = [f for f in features_to_90 if f not in to_remove]

#Listado de features para 90min:
#features_to_90 = ['ire_90', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_90']


In [208]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5']


Guardamos los listados de features:

In [209]:
import json

features_dict = {
    #"features_to_30": features_to_30,
    #"features_to_60": features_to_60,
    "features_to_90": features_to_90
}

with open(f'{drive_path}/5_transformer_90_model/features_list.json', 'w') as f:
    json.dump(features_dict, f, indent=4)


In [210]:
##Para abrir desde otra notebook:
'''
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]
'''

'\nimport json\n\n# Ruta al archivo guardado\npath = f\'{drive_path}/2_feature_engineering/features_list.json\'\n\nwith open(path, "r") as f:\n    features_dict = json.load(f)\n\n# Extraer las listas\nfeatures_to_30 = features_dict["features_to_30"]\nfeatures_to_60 = features_dict["features_to_60"]\nfeatures_to_90 = features_dict["features_to_90"]\n'

## 5. Preparamos el dataset mnq_to_model


Combinamos los tres listados para obtener un listado total de features

In [211]:
# Combinar los tres listados y eliminar duplicados
#features_to_model = list(set(features_to_30 + features_to_60 + features_to_90))
features_to_model = list(set(features_to_90))
features_to_model.sort()

In [212]:
print(f'Listado de features para los modelos: {features_to_model}')

Listado de features para los modelos: ['bb_60', 'ire_90', 'momentum_5', 'rev_mom_z_90', 'roc_60']


Finalmente filtramos el dataset, solo para conservar las columnas de interés:

In [213]:
# Dataset a filtrar: mnq_features_combined
# A mantener: columnas_base + columnas_target + features_to_model
# Dataset final: mnq_to_model

cols_to_keep = columnas_base + columnas_target + features_to_model
mnq_to_model = mnq_features_combined[cols_to_keep]


In [214]:
mnq_to_model

,date,open,high,low,close,volume,target_return_90,bb_60,ire_90,momentum_5,rev_mom_z_90,roc_60
datetime,,,,,,,,,,,,
2019-12-23 06:00:00-05:00,2019-12-23,8719.50,8719.50,8719.50,8719.50,17,0.001976,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:01:00-05:00,2019-12-23,8720.00,8720.00,8719.50,8720.00,12,0.001805,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:02:00-05:00,2019-12-23,8720.00,8720.75,8720.00,8720.75,26,0.001633,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:03:00-05:00,2019-12-23,8720.75,8720.75,8720.00,8720.50,23,0.001604,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:04:00-05:00,2019-12-23,8720.25,8720.25,8720.00,8720.25,7,0.001576,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,NaN,0.100025,-1.773812,-0.000947,1.223657,-0.203125
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,NaN,0.191181,-1.683865,-0.000474,0.903176,-0.182336
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,NaN,0.175104,-1.712648,-0.000035,0.700377,-0.102800


In [215]:
# Guardar como Parquet
mnq_to_model.to_parquet(f'{drive_path}/5_transformer_90_model/mnq_to_model.parquet')